# 1002 · 03 · Corregir la cuarentena y reintegrar en silver

La validación ha dejado en `cuarentena` las filas que incumplen alguna regla, cada una con su `motivo`. Este notebook:

1. añade columnas de **trazabilidad** (`fixed`, `type_fix`, `date_fix`, `fix_version`);
2. aplica **reglas de corrección** acordadas con negocio;
3. **reintegra** en silver lo corregido y lo borra de la cuarentena;
4. deja a la vista lo que sigue pendiente, que necesita una decisión humana.

Se ejecuta a mano la primera vez para entenderlo y, después, como **última tarea del job**. Por eso el catálogo llega como parámetro.

In [ ]:
dbutils.widgets.text("catalogo", "lab1002")
CATALOGO = dbutils.widgets.get("catalogo")
Q = f"{CATALOGO}.cuarentena"   # tablas de cuarentena
S = f"{CATALOGO}.silver"       # tablas validas

print("Catálogo:", CATALOGO)

## 0 · Qué hay en cuarentena

In [ ]:
display(spark.sql(f"""
  SELECT 'dim_aircraft' AS tabla, motivo, count(*) AS filas FROM {Q}.dim_aircraft GROUP BY motivo
  UNION ALL
  SELECT 'fact_engine_sensor', motivo, count(*) FROM {Q}.fact_engine_sensor GROUP BY motivo
  ORDER BY tabla, filas DESC
"""))

## 1 · Columnas de trazabilidad

Cada corrección queda registrada: si la fila está arreglada, con qué regla, cuándo y en qué versión. Así la cuarentena guarda también el historial de lo que se ha tocado.

In [ ]:
def anadir_trazabilidad(tabla):
    existentes = {c.lower() for c in spark.table(tabla).columns}
    nuevas = [d for d in ["fixed BOOLEAN", "type_fix STRING", "date_fix TIMESTAMP", "fix_version INT"]
              if d.split()[0] not in existentes]
    if nuevas:
        spark.sql(f"ALTER TABLE {tabla} ADD COLUMNS ({', '.join(nuevas)})")

for t in ["dim_aircraft", "fact_engine_sensor"]:
    anadir_trazabilidad(f"{Q}.{t}")

## 2 · `dim_aircraft`

### 2.1 Boeing sin horas de vuelo

Los Boeing de la flota hacen vuelos nacionales, con ciclos de entre 1 h 30 y 2 h 15: de media, **2 horas por ciclo**. Negocio acuerda que, si `hours_total` falta o es negativo, se estime como `cycles_total × 2`.

In [ ]:
spark.sql(f"""
  UPDATE {Q}.dim_aircraft
  SET hours_total = cycles_total * 2,
      fixed = true, type_fix = 'hours_from_cycles', date_fix = current_timestamp(), fix_version = 1
  WHERE manufacturer = 'Boeing'
    AND (hours_total IS NULL OR hours_total < 0)
    AND cycles_total IS NOT NULL
""")
display(spark.table(f"{Q}.dim_aircraft"))

### 2.2 Motor vacío

Todas las aeronaves de un mismo modelo llevan el mismo motor. Si `engine_model` está vacío, se toma el que tienen en silver las demás aeronaves de su modelo.

In [ ]:
spark.sql(f"""
  CREATE OR REPLACE TEMP VIEW motor_por_modelo AS
  SELECT model, first(engine_model, true) AS engine_model
  FROM {S}.dim_aircraft
  WHERE engine_model IS NOT NULL AND trim(engine_model) <> ''
  GROUP BY model
""")

spark.sql(f"""
  MERGE INTO {Q}.dim_aircraft AS q
  USING motor_por_modelo AS m
  ON q.model = m.model
  WHEN MATCHED AND (q.engine_model IS NULL OR trim(q.engine_model) = '') THEN UPDATE SET
    q.engine_model = m.engine_model,
    q.fixed = true, q.type_fix = 'engine_from_model', q.date_fix = current_timestamp(),
    q.fix_version = coalesce(q.fix_version, 0) + 1
""")
display(spark.table(f"{Q}.dim_aircraft"))

## 3 · Reintegrar en silver

Lo que está marcado como `fixed` vuelve a silver con un `MERGE` (actualiza si ya existe, inserta si no) y sale de la cuarentena. La función sirve para cualquier tabla: toma las columnas de silver, así que las de trazabilidad no se cuelan.

In [ ]:
def reintegrar(tabla, clave):
    columnas = ", ".join(spark.table(f"{S}.{tabla}").columns)
    spark.sql(f"""
      MERGE INTO {S}.{tabla} AS s
      USING (SELECT {columnas} FROM {Q}.{tabla} WHERE fixed = true) AS q
      ON s.{clave} = q.{clave}
      WHEN MATCHED THEN UPDATE SET *
      WHEN NOT MATCHED THEN INSERT *
    """)
    n = spark.sql(f"DELETE FROM {Q}.{tabla} WHERE fixed = true").first()[0]
    print(f"{tabla}: {n} filas reintegradas en silver")

reintegrar("dim_aircraft", "aircraft_id")

## 4 · `fact_engine_sensor`

### 4.1 Altitud multiplicada por 10

Los B737-800 llevan un registrador en pruebas (`source_system = RECORDER_TEST`) que añade un cero a la altitud. Por encima de 50.000 pies no vuela ningún avión de la flota, así que en un B737-800 esa lectura es el fallo conocido: se divide entre 10.

In [ ]:
spark.sql(f"""
  MERGE INTO {Q}.fact_engine_sensor AS q
  USING {S}.dim_aircraft AS a
  ON q.aircraft_id = a.aircraft_id
  WHEN MATCHED AND a.model = 'B737-800' AND q.altitude_ft > 50000 THEN UPDATE SET
    q.altitude_ft = CAST(q.altitude_ft / 10 AS INT),
    q.fixed = true, q.type_fix = 'altitude_x10_b737', q.date_fix = current_timestamp(), q.fix_version = 1
""")
display(spark.table(f"{Q}.fact_engine_sensor"))

### 4.2 Velocidad en km/h en vez de nudos

El sistema antiguo (`source_system = ACMS_LEGACY`) envía la velocidad indicada en **km/h** durante las fases `CLIMB` y `DESCENT`. En nudos, ningún avión pasa de 600: se convierte dividiendo entre 1,852.

In [ ]:
spark.sql(f"""
  UPDATE {Q}.fact_engine_sensor
  SET ias_kts = CAST(round(ias_kts / 1.852) AS INT),
      fixed = true, type_fix = 'ias_kmh_to_kts', date_fix = current_timestamp(),
      fix_version = coalesce(fix_version, 0) + 1
  WHERE ias_kts > 600
    AND upper(phase_of_flight) IN ('CLIMB', 'DESCENT')
""")

reintegrar("fact_engine_sensor", "reading_id")

## 5 · Lo que sigue pendiente

Estas filas no tienen una regla de corrección: una clave duplicada o una altitud de 62.000 pies en un A350 necesitan que alguien decida. Se quedan en cuarentena, con su motivo, a la espera.

In [ ]:
display(spark.sql(f"""
  SELECT 'dim_aircraft' AS tabla, motivo, count(*) AS filas FROM {Q}.dim_aircraft GROUP BY motivo
  UNION ALL
  SELECT 'fact_engine_sensor', motivo, count(*) FROM {Q}.fact_engine_sensor GROUP BY motivo
  ORDER BY tabla, filas DESC
"""))